# Parte IV: Sistemas de Recomendación Híbridos

Este notebook corre de forma autónoma la Parte IV del proyecto.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
# Setup inicial de importaciones
import time
import pandas as pd
import json
import numpy as np
from implementaciones.preprocessing import load_reviews_efficiently, load_business_efficiently


In [3]:
import pandas as pd

REVIEW_PATH = 'Yelp-JSON/Yelp JSON/yelp_academic_dataset_review_10core.json'
BUSINESS_PATH = 'Yelp-JSON/Yelp JSON/yelp_academic_dataset_business.json'

print("1. Cargando reviews (10-Core con Hashes originales)...")
sample_df = pd.read_json(REVIEW_PATH, lines=True)

print("2. Cargando catálogo de negocios...")
df_business = pd.read_json(BUSINESS_PATH, lines=True)

# =====================================================================
# CONSTRUCCIÓN UNIFICADA DE MAPAS (El estándar de la industria)
# =====================================================================
print("3. Generando mapeadores dinámicos unificados...")

# Creamos el mapa oficial usando el orden único de las REVIEWS
# Esto garantiza que el entero '9344' signifique exactamente lo mismo en todo el notebook
business_unicos_reviews = sample_df['business_id'].unique()
item_mapping = {hash_id: idx for idx, hash_id in enumerate(business_unicos_reviews)}

# Mapeamos las reviews a enteros para los modelos
sample_df['business_id_numeric'] = sample_df['business_id'].map(item_mapping)

# Mapeamos el catálogo de negocios usando el MISMO diccionario exacto
df_business['business_id_numeric'] = df_business['business_id'].map(item_mapping)

# Eliminamos del catálogo los negocios que no están en nuestra muestra de reviews
df_business_filtrado = df_business.dropna(subset=['business_id_numeric']).copy()
df_business_filtrado['business_id_numeric'] = df_business_filtrado['business_id_numeric'].astype(int)

# Construimos el business_map final indexado por el ENTERO correcto
business_map = df_business_filtrado.set_index('business_id').to_dict('index')

print(f"¡Sincronización exitosa! Universo de negocios acoplado: {len(business_map)}")

1. Cargando reviews (10-Core con Hashes originales)...
2. Cargando catálogo de negocios...
3. Generando mapeadores dinámicos unificados...
¡Sincronización exitosa! Universo de negocios acoplado: 53085


# Parte IV: Sistemas de Recomendación Híbridos
Construir sistemas de recomendación escalables. Integrar múltiples enfoques para mejorar precisión y cobertura.

In [4]:
from implementaciones.recommenders import CollaborativeFiltering, ContentBasedFiltering, HybridRecommender, TFIDF
from implementaciones.recommenders import RandomRecommender, PopularityRecommender
from implementaciones.recommenders import precision_at_k, recall_at_k, ndcg_at_k, rmse, mae, novelty, diversity
import numpy as np

print("Construyendo matriz Usuario-Item dispersa (usando todo el sampleo)...")
unique_users = sample_df['user_id'].unique()
unique_items = sample_df['business_id'].unique()

user_mapping = {u: i for i, u in enumerate(unique_users)}
item_mapping = {b: i for i, b in enumerate(unique_items)}

U_I_matrix = {}
for _, row in sample_df.iterrows():
    u = user_mapping[row['user_id']]
    i = item_mapping[row['business_id']]
    if u not in U_I_matrix:
        U_I_matrix[u] = {}
    U_I_matrix[u][i] = row['stars']

print(f"Matriz Usuario-Item (Dispersa) creada: {len(U_I_matrix)} usuarios, {len(unique_items)} items")

Construyendo matriz Usuario-Item dispersa (usando todo el sampleo)...
Matriz Usuario-Item (Dispersa) creada: 28313 usuarios, 53085 items


### 1. Extracción de Features (Content-Based) usando TF-IDF
Se implementó una clase `TFIDF` propia que extrae los atributos (categorías de productos) de cada ítem y genera el vector ponderado en espacio de features.

In [5]:
print("Extrayendo Features de Contenido para TODOS los items en el sampleo...")
# Usar business_map para búsquedas en O(1) de manera ultra eficiente
corpus = []
for item_id_str in unique_items:
    biz = business_map.get(item_id_str, {})
    tokens = []
    
    # 1. Añadir Categorías
    cats = biz.get('categories')
    if cats and isinstance(cats, str):
        tokens.extend([c.strip() for c in cats.split(',') if c.strip()])
        
    # 2. Añadir Atributos (Extrae las llaves de los atributos como texto)
    attrs = biz.get('attributes')
    if attrs and isinstance(attrs, dict):
        # Añade atributos que sean True o texto descriptivo
        for k, v in attrs.items():
            if v == 'True' or v is True:
                tokens.append(k)
            elif isinstance(v, str) and v != 'False':
                tokens.append(f"{k}_{v}") # Ej: "RestaurantsPriceRange2_2"
                
    corpus.append(tokens)

tfidf = TFIDF()
I_features = tfidf.fit_transform(corpus)
print(f"Matriz TF-IDF (Items x Features) creada con dimensiones: {I_features.shape}")

Extrayendo Features de Contenido para TODOS los items en el sampleo...
Matriz TF-IDF (Items x Features) creada con dimensiones: (53085, 4483)


### 2. Filtrado Colaborativo (User-Based/ Item-Based)
Se calcula la similitud Usuario-Usuario usando métricas como Coseno y Pearson. Luego se recomiendan ítems usando k-NN ponderado.

In [6]:
print("--- Inicializando Modelos de Filtrado Colaborativo ---")

user_id = 0

cf_cosine = CollaborativeFiltering(based='user', similarity_metric='cosine')
cf_cosine.fit(U_I_matrix)

cf_pearson = CollaborativeFiltering(based='user', similarity_metric='pearson')
cf_pearson.fit(U_I_matrix)

top_k_cf_cos = cf_cosine.predict(user_id, top_k=5, k_nn=10)
top_k_cf_pear = cf_pearson.predict(user_id, top_k=5, k_nn=10)
print("user - user")
print(f"Recomendaciones CF (Coseno) para usuario {user_id}: {top_k_cf_cos}")
print(f"Recomendaciones CF (Pearson) para usuario {user_id}: {top_k_cf_pear}")

cf_cosine_ = CollaborativeFiltering(based='item', similarity_metric='cosine')
cf_cosine_.fit(U_I_matrix)

cf_pearson_ = CollaborativeFiltering(based='item', similarity_metric='pearson')
cf_pearson_.fit(U_I_matrix)

top_k_cf_cos_ = cf_cosine_.predict(user_id, top_k=5, k_nn=10)
top_k_cf_pear_ = cf_pearson_.predict(user_id, top_k=5, k_nn=10)
print("item - item")
print(f"Recomendaciones CF (Coseno) para usuario {user_id}: {top_k_cf_cos_}")
print(f"Recomendaciones CF (Pearson) para usuario {user_id}: {top_k_cf_pear_}")

--- Inicializando Modelos de Filtrado Colaborativo ---
user - user
Recomendaciones CF (Coseno) para usuario 0: [8562, 20060, 1637, 11085, 13362]
Recomendaciones CF (Pearson) para usuario 0: [3474, 8562, 11096, 9275, 1423]
item - item
Recomendaciones CF (Coseno) para usuario 0: [18662, 21789, 26526, 30588, 32744]
Recomendaciones CF (Pearson) para usuario 0: [2754, 4075, 5596, 8674, 9225]


### Análisis del Problema de "Cold-Start" (Arranque en Frío)

El **Cold-Start** ocurre cuando ingresa un nuevo usuario o un nuevo producto al sistema, careciendo de interacciones históricas.
1. **Usuarios Nuevos:** El *Filtrado Colaborativo Puro (CF)* fracasa por completo porque no puede encontrar vecinos similares al no tener calificaciones anteriores (la distancia coseno contra matrices llenas de ceros falla). Nuestro **Modelo Híbrido** mitiga esto porque, si no hay ratings, el CF devuelve cero, pero podemos pedirle al usuario sus preferencias iniciales e inferir su *user_profile* (basado en características o categorías como vimos en el Content-Based), permitiéndole al modelo Híbrido realizar recomendaciones usando el peso del `cb_weight` o recurrir al modelo *Baseline de Popularidad*.
2. **Productos Nuevos:** Si un negocio nuevo se registra en Yelp, el CF jamás lo va a recomendar. Sin embargo, al extraer las características del nuevo negocio usando **TF-IDF** para incorporarlo a nuestra matriz `I_features`, el módulo Content-Based podrá encontrar de forma inmediata similitudes matemáticas con los perfiles históricos de los usuarios e ingresarlo exitosamente al pipeline de recomendaciones antes de que reciba su primera reseña.

### 3. Filtrado basado en contenido (Content-Based)
Se calcula la similitud entre el perfil del usuario (basado en las características de los ítems que ha calificado) y las características de los ítems nuevos usando la matriz `I_features` generada por TF-IDF. Se recomienda el top-N ítems con mayor similitud.

In [7]:
# ==========================================
# 1.5 Prueba de Content-Based Filtering
# ==========================================

print("Entrenando recomendador basado en contenido...")
cb = ContentBasedFiltering()
cb.fit(I_features) # I_features es la matriz obtenida con TF-IDF

# Seleccionamos un usuario de prueba (ej. usuario 0)
target_user_idx = 15
user_ratings = U_I_matrix.get(target_user_idx, {})

# Construir Perfil de Usuario (Promedio de las features de los ítems que le gustaron)
# Filtramos ítems con rating alto (ej. >= 4 estrellas)
liked_items = [item for item, rating in user_ratings.items() if rating >= 1]

if len(liked_items) > 0:
    # El perfil es el centroide (promedio) de las características de sus negocios favoritos
    user_profile = np.mean(I_features[liked_items], axis=0)
    
    # Predecir top 5 recomendaciones
    cb_recs = cb.predict(user_profile, top_k=5)
    
    print(f"Perfil del usuario {target_user_idx} creado basado en {len(liked_items)} ítems que le gustaron.")
    print(f"Top 5 Recomendaciones Content-Based (Índices): {cb_recs}")
    
    # (Opcional) Ver los IDs reales de los negocios recomendados
    for rank, idx in enumerate(cb_recs, 1):
        business_id = unique_items[idx]
        print(f"{rank}. {business_id}")
else:
    print(f"\nEl usuario {target_user_idx} no tiene suficientes calificaciones positivas para crear un perfil.")

Entrenando recomendador basado en contenido...
Perfil del usuario 15 creado basado en 11 ítems que le gustaron.
Top 5 Recomendaciones Content-Based (Índices): [20818 15999 35415 30955 23134]
1. wo6RAA4c8I5trN331y8SIw
2. nnn78cplvQGZSMVblpI5gQ
3. ABC7RM6MVsM1QMOX84UKiQ
4. JvcLVPXUj8fOJ9yuLf8HjA
5. mE16tq2q9kIAeI1wnPkKXQ


### 4. Modelo Híbrido y Baselines
Combinaremos las predicciones del modelo CF (User-Based) con las del modelo CBF mediante un promedio ponderado (*Weighted Average*). Utilizaremos un *Random Baseline* y un *Popularity Baseline* para su contrastación.

### Evaluación 
Se probarán los métodos en la extracción de las métricas **Precision@K**, **Recall@K**, **NDCG**, **RMSE** y **MAE**.

In [10]:
from collections import defaultdict
print("--- Evaluación Rigurosa de Modelos ---")

# ==========================================
# PARÁMETROS DE CONFIGURACIÓN DE EVALUACIÓN
# ==========================================
K_RECS = 30          # Tamaño de la lista de recomendación (Top-K)
KNN_HYBRID = 20       # Número de vecinos más cercanos para el Collaborative Filtering del híbrido
CF_WEIGHT = 0.85      # Peso asignado al Collaborative Filtering (0.0 a 1.0)
CB_WEIGHT = 0.15       # Peso asignado al Content-Based (0.0 a 1.0)
# ==========================================

metrics_hyb = {'p':[], 'r':[], 'ndcg':[], 'rmse':[], 'mae':[], 'nov':[], 'div':[]}
metrics_pop = {'p':[], 'r':[], 'ndcg':[], 'rmse':[], 'mae':[], 'nov':[], 'div':[]}
metrics_rnd = {'p':[], 'r':[], 'ndcg':[], 'rmse':[], 'mae':[], 'nov':[], 'div':[]}

# Splitting train and test for the evaluated users para evitar el problema de intersección nula
np.random.seed(42)
total_users = len(U_I_matrix)
U_I_train = {u: items.copy() for u, items in U_I_matrix.items()}
U_I_test = defaultdict(dict)

# Contar calificaciones totales por item en U_I_matrix para evitar el problema de cold-start absoluto de items de prueba
item_counts = defaultdict(int)
for u, items in U_I_matrix.items():
    for item in items:
        item_counts[item] += 1
        
eval_users = [u for u, items in U_I_matrix.items() if len(items) >= 4][:100]

for u_id in eval_users:
    # Solo ocultamos items en test si tienen al menos 5 calificaciones en total en el dataset
    positives = [item for item, rating in U_I_matrix[u_id].items() if rating >= 3.5 and item_counts[item] >= 5]
    if len(positives) >= 2:
        num_hide = max(1, int(len(positives) * 0.3))
        hidden = np.random.choice(positives, num_hide, replace=False)
        for item in hidden:
            U_I_train[u_id].pop(item)
            U_I_test[u_id][item] = U_I_matrix[u_id][item]

# Re-train models on Train split (para no sugerir items que sí estaban en test)
eval_hybrid = HybridRecommender(cf_weight=CF_WEIGHT, cb_weight=CB_WEIGHT, based='user', similarity_metric='cosine')
eval_hybrid.fit(U_I_train, I_features)

eval_pop = PopularityRecommender()
eval_pop.fit(U_I_train)

eval_rnd = RandomRecommender()
eval_rnd.fit(U_I_train)

for u_id in eval_users:
    test_items = list(U_I_test[u_id].keys()) if u_id in U_I_test else []
    if len(test_items) == 0: continue
    
    # User profile based strictly on train interactions
    train_items = list(U_I_train[u_id].keys()) if u_id in U_I_train else []
    if len(train_items) == 0:
        user_profile = np.zeros(I_features.shape[1])
    else:
        # Tomamos las calificaciones del usuario para esos ítems y las centramos en la mitad (3.0)
        ratings = np.array([U_I_train[u_id][item] - 3.0 for item in train_items])
        
        # Multiplicamos las características de cada ítem por su peso/calificación
        weighted_features = I_features[train_items] * ratings[:, np.newaxis]
        
        # Sumamos en lugar de promediar para mantener la intensidad de las palabras clave
        user_profile = np.sum(weighted_features, axis=0)
        
        # Normalizamos el vector para evitar valores gigantescos
        if np.linalg.norm(user_profile) > 0:
            user_profile = user_profile / np.linalg.norm(user_profile)
        
    # Top-K Predictions (se pasa explícitamente KNN_HYBRID y K_RECS)
    preds_hyb = eval_hybrid.predict(u_id, user_profile, top_k=K_RECS, k_nn=KNN_HYBRID)
    preds_pop = eval_pop.predict(u_id, U_I_train, top_k=K_RECS)
    preds_rnd = eval_rnd.predict(u_id, top_k=K_RECS)
    
    actual_relevant_items = test_items
    
    metrics_hyb['p'].append(precision_at_k(actual_relevant_items, preds_hyb, k=K_RECS))
    metrics_hyb['r'].append(recall_at_k(actual_relevant_items, preds_hyb, k=K_RECS))
    metrics_hyb['ndcg'].append(ndcg_at_k(actual_relevant_items, preds_hyb, k=K_RECS))
    metrics_hyb['nov'].append(novelty(preds_hyb, item_counts, total_users))
    metrics_hyb['div'].append(diversity(preds_hyb, I_features))
    
    metrics_pop['p'].append(precision_at_k(actual_relevant_items, preds_pop, k=K_RECS))
    metrics_pop['r'].append(recall_at_k(actual_relevant_items, preds_pop, k=K_RECS))
    metrics_pop['ndcg'].append(ndcg_at_k(actual_relevant_items, preds_pop, k=K_RECS))
    metrics_pop['nov'].append(novelty(preds_pop, item_counts, total_users))
    metrics_pop['div'].append(diversity(preds_pop, I_features))
    
    metrics_rnd['p'].append(precision_at_k(actual_relevant_items, preds_rnd, k=K_RECS))
    metrics_rnd['r'].append(recall_at_k(actual_relevant_items, preds_rnd, k=K_RECS))
    metrics_rnd['ndcg'].append(ndcg_at_k(actual_relevant_items, preds_rnd, k=K_RECS))
    metrics_rnd['nov'].append(novelty(preds_rnd, item_counts, total_users))
    metrics_rnd['div'].append(diversity(preds_rnd, I_features))
    
    # Rating Predictions (RMSE, MAE)
    actual_r = [U_I_test[u_id][item] for item in test_items]
    
    pred_r_hyb = [eval_hybrid.predict_rating(u_id, user_profile, i, k_nn=KNN_HYBRID) for i in test_items]
    pred_r_pop = [eval_pop.predict_rating(u_id, i) for i in test_items]
    pred_r_rnd = [eval_rnd.predict_rating(u_id, i) for i in test_items]
    
    metrics_hyb['rmse'].append(rmse(actual_r, pred_r_hyb))
    metrics_hyb['mae'].append(mae(actual_r, pred_r_hyb))
    
    metrics_pop['rmse'].append(rmse(actual_r, pred_r_pop))
    metrics_pop['mae'].append(mae(actual_r, pred_r_pop))
    
    metrics_rnd['rmse'].append(rmse(actual_r, pred_r_rnd))
    metrics_rnd['mae'].append(mae(actual_r, pred_r_rnd))

print("Resultados Promedio (usando Test Split):")
print(f"{'Modelo':<15} | {f'Prec@{K_RECS}':<8} | {f'Rec@{K_RECS}':<8} | {f'NDCG@{K_RECS}':<8} | {'RMSE':<8} | {'MAE':<8} | {'Novelty':<8} | {'Diversity':<8}")
print("-" * 100)
print(f"{'Híbrido':<15} | {np.mean(metrics_hyb['p']):.4f}   | {np.mean(metrics_hyb['r']):.4f}   | {np.mean(metrics_hyb['ndcg']):.4f}   | {np.mean(metrics_hyb['rmse']):.4f}   | {np.mean(metrics_hyb['mae']):.4f} | {np.mean(metrics_hyb['nov']):.4f}   | {np.mean(metrics_hyb['div']):.4f}")
print(f"{'Popularidad':<15} | {np.mean(metrics_pop['p']):.4f}   | {np.mean(metrics_pop['r']):.4f}   | {np.mean(metrics_pop['ndcg']):.4f}   | {np.mean(metrics_pop['rmse']):.4f}   | {np.mean(metrics_pop['mae']):.4f} | {np.mean(metrics_pop['nov']):.4f}   | {np.mean(metrics_pop['div']):.4f}")
print(f"{'Random':<15} | {np.mean(metrics_rnd['p']):.4f}   | {np.mean(metrics_rnd['r']):.4f}   | {np.mean(metrics_rnd['ndcg']):.4f}   | {np.mean(metrics_rnd['rmse']):.4f}   | {np.mean(metrics_rnd['mae']):.4f} | {np.mean(metrics_rnd['nov']):.4f}   | {np.mean(metrics_rnd['div']):.4f}")

--- Evaluación Rigurosa de Modelos ---
Resultados Promedio (usando Test Split):
Modelo          | Prec@30  | Rec@30   | NDCG@30  | RMSE     | MAE      | Novelty  | Diversity
----------------------------------------------------------------------------------------------------
Híbrido         | 0.0040   | 0.0326   | 0.0182   | 1.2535   | 1.1740 | 9.8864   | 0.8185
Popularidad     | 0.0040   | 0.0333   | 0.0168   | 0.8373   | 0.7671 | 6.0962   | 0.8520
Random          | 0.0000   | 0.0000   | 0.0000   | 1.9848   | 1.7911 | 11.7268   | 0.9150
